## Spark vs Pandas vs Pandas-on-Spark

### Initialize Spark Session

In [4]:
# Suppress native-hadoop warning
!sed -i '$a\# Add the line for suppressing the NativeCodeLoader warning \nlog4j.logger.org.apache.hadoop.util.NativeCodeLoader=ERROR,console' /$HADOOP_HOME/etc/hadoop/log4j.properties
import os; os.close(os.dup2(os.open(os.devnull, os.O_WRONLY), 2))

In [5]:
import os
# Requried by Pandas-on-Spark
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

import pandas as pd
import pyspark
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, Row
import pyspark.pandas as ps
import pandas as pd


In [6]:
conf = pyspark.SparkConf().setAll([('spark.master', 'local[4]'),
                                   ('spark.app.name', 'Pandas-on-spark demo')])
spark = SparkSession.builder.config(conf=conf).getOrCreate()
pyspark.__version__

'4.1.1'

### Read data
Set the path for the data file 'minute_weather_modified.csv'

In [7]:
!pwd


/home/work/2026/S4/Demo4B-Pandas-on-Spark


In [8]:
file_path = 'file:///home/work/2026/S4/Demo4B-Pandas-on-Spark/minute_weather_modified.csv' #file URI

#### Spark
Read the data file into a PySpark DataFrame, inferring the schema and reading in the column headers

In [9]:
sdf = spark.read.csv(file_path, inferSchema=True, header=True)

#### Pandas
Read the data file into a Pandas DataFrame

In [10]:
pdf = pd.read_csv(file_path)

#### Pandas-on-spark
Read the data file into a PySpark DataFrame using the Pandas API, using rowID column for index

In [11]:
psdf = ps.read_csv(file_path, index_col='rowID')

## Schema of the dataframe (For reference - Spark only)

Print the schema of the PySpark DataFrame

In [12]:
sdf.printSchema()

root
 |-- rowID: integer (nullable = true)
 |-- hpwren_timestamp: timestamp (nullable = true)
 |-- air_pressure: double (nullable = true)
 |-- air_temp: double (nullable = true)
 |-- avg_wind_direction: double (nullable = true)
 |-- avg_wind_speed: double (nullable = true)
 |-- max_wind_direction: double (nullable = true)
 |-- max_wind_speed: double (nullable = true)
 |-- min_wind_direction: double (nullable = true)
 |-- min_wind_speed: double (nullable = true)
 |-- rain_accumulation: double (nullable = true)
 |-- rain_duration: double (nullable = true)
 |-- relative_humidity: double (nullable = true)
 |-- air_temp_category: string (nullable = true)



## Drop null values

### How many rows have null values in at least one column?
Get the number of rows, then drop NAs, then find the difference in number of rows after dropping NAs

**Useful functions:**

Spark:

[pyspark.sql.DataFrame.count](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.count.html)

[pyspark.sql.DataFrame.dropna](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropna.html)

Pandas:

[pandas.DataFrame.dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)

Pandas-on-Spark:

[pyspark.pandas.DataFrame.dropna](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.dropna.html)

**NOTE:** To get the total number of rows in a Pandas or Pandas-on-Spark dataframe, you can use len() or df.shape\[0\].

#### Spark

In [13]:

init_len = sdf.count()
sdf = sdf.dropna()
init_len - sdf.count()

434

#### Pandas

In [14]:

init_len = len(pdf)
pdf.dropna(inplace=True)
init_len - len(pdf)

434

#### Pandas-on-spark

In [15]:

init_len = len(psdf)
psdf.dropna(inplace=True)
init_len - len(psdf)

434

## Count the number of rows

#### Spark

In [16]:
#Error if do len(sdf) or sdf.shape[0]
sdf.count()

1586823

#### Pandas

In [17]:
len(pdf)

1586823

In [18]:
pdf.shape[0]

1586823

#### Pandas-on-spark

In [19]:
len(psdf)

1586823

In [20]:
psdf.shape[0]

1586823

In [21]:
pdf.count() #same as psdf.count()

rowID                 1586823
hpwren_timestamp      1586823
air_pressure          1586823
air_temp              1586823
avg_wind_direction    1586823
avg_wind_speed        1586823
max_wind_direction    1586823
max_wind_speed        1586823
min_wind_direction    1586823
min_wind_speed        1586823
rain_accumulation     1586823
rain_duration         1586823
relative_humidity     1586823
air_temp_category     1586823
dtype: int64

## Summary statistics

### Show the min, max, and mean value of the column `air_temp`

**Useful functions:**

Spark:

[pyspark.sql.functions.avg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.avg.html) (same as [pyspark.sql.functions.mean](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.mean.html))

[pyspark.sql.functions.min](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.min.html)

[pyspark.sql.functions.max](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.max.html)

[pyspark.sql.DataFrame.select](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html)

[pyspark.sql.Column.alias](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Column.alias.html)

Pandas:

[pandas.Series.agg](https://pandas.pydata.org/docs/reference/api/pandas.Series.agg.html)

Pandas-on-Spark:

[pyspark.pandas.Series.agg](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.Series.agg.html)


In [22]:
from pyspark.sql import functions as F

#### Spark

In [23]:

sdf.select(F.avg('air_temp').alias('Mean'),
           F.min('air_temp').alias('Min'),
           F.max('air_temp').alias('Max')).show()

[Stage 25:===========================================>              (3 + 1) / 4]

+-----------------+-----+----+
|             Mean|  Min| Max|
+-----------------+-----+----+
|61.85526703355015|31.64|99.5|
+-----------------+-----+----+



#### Pandas

In [24]:
pdf['air_temp'].agg(['min','max','mean'])

# pdf[['air_temp']].agg(['min','max', 'mean']) #to show as a dataframe instead of a series

min     31.640000
max     99.500000
mean    61.855267
Name: air_temp, dtype: float64

#### Pandas-on-spark

In [25]:
psdf['air_temp'].agg(['min','max', 'mean'])

# psdf[['air_temp']].agg(['min','max', 'mean']) #to show as a dataframe instead of a series

min     31.640000
max     99.500000
mean    61.855267
Name: air_temp, dtype: float64

## Show the min, max, and mean of `avg_wind_speed` for values of `avg_wind_direction` between 0 and 90 (inclusive):
* Filter by values of `avg_wind_direction` and calculate the min, max, and mean of `avg_wind_speed`

#### Spark

In [26]:

sdf1 = sdf.filter((sdf.avg_wind_direction >= 0) & (sdf.avg_wind_direction <= 90))
sdf1.select(F.avg('avg_wind_speed'), F.min('avg_wind_speed'), F.max('avg_wind_speed')).show()

[Stage 31:==========================================================(4 + 0) / 4]

+-------------------+-------------------+-------------------+
|avg(avg_wind_speed)|min(avg_wind_speed)|max(avg_wind_speed)|
+-------------------+-------------------+-------------------+
|  2.783366067381772|                0.0|               20.7|
+-------------------+-------------------+-------------------+



#### Pandas

In [27]:

pdf[(pdf['avg_wind_direction'] >= 0) & (pdf['avg_wind_direction'] <= 90)]['avg_wind_speed'].agg(['min', 'max', 'mean'])

min      0.000000
max     20.700000
mean     2.783366
Name: avg_wind_speed, dtype: float64

#### Pandas-on-spark

In [28]:

psdf[(psdf['avg_wind_direction'] >= 0) & (psdf['avg_wind_direction'] <= 90)]['avg_wind_speed'].agg(['min', 'max', 'mean'])

min      0.000000
max     20.700000
mean     2.783366
Name: avg_wind_speed, dtype: float64

### Filtering Followed by GroupBy
Filter rows with 'avg_wind_direction' between 0 and 90 (inclusive). 
Then, use groupBy to count the number of records for each unique value of 'air_temp_category' ('High', 'Low', 'Medium')

**Useful Functions:** 

Spark:

[pyspark.sql.DataFrame.groupBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.groupBy.html)

Pandas:

[pandas.DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

Pandas-on-Spark:

[pyspark.pandas.DataFrame.groupby](https://spark.apache.org/docs/latest/api/python/reference/pyspark.pandas/api/pyspark.pandas.DataFrame.groupby.html)

#### Spark

In [29]:
sdf1.groupBy(F.col('air_temp_category')).count().show()

[Stage 37:=============================>                            (2 + 2) / 4]

+-----------------+------+
|air_temp_category| count|
+-----------------+------+
|             High|  3618|
|              Low| 63128|
|           Medium|416441|
+-----------------+------+



#### Pandas

In [30]:
# Create a copy so original DF is not changed
pdf1 = pdf[(pdf['avg_wind_direction'] >= 0) & (pdf['avg_wind_direction'] <= 90)].copy()
pdf1.groupby('air_temp_category')["air_temp_category"].count()


air_temp_category
High        3618
Low        63128
Medium    416441
Name: air_temp_category, dtype: int64

#### Pandas-on-Spark

In [31]:

psdf1 = psdf[(psdf['avg_wind_direction'] >= 0) & (psdf['avg_wind_direction'] <= 90)]
psdf1.groupby('air_temp_category').air_temp_category.count()

air_temp_category
High        3618
Low        63128
Medium    416441
Name: air_temp_category, dtype: int64

### Stop Spark Session

In [32]:
spark.stop()